# Actually, copy repo cos CLOSE_ORIGINAL solves all our problems about accessing PIT Price for scanning.
## Adjusting for Splits and Dividends - The correct way
#### You should treat cash dividends differently than splits, especially for the timeframes you are trading (intraday).
Massive support: "For your question on splits vs dividends -- companies may issue dividends as form of stocks or cash (a.k.a. stock dividends vs cash dividends). When a stocks dividend happens (as you see from our split endpoint with adjustment_type=stock_dividend), those events impact stock's price and should be documented as splits; The cash dividends on the other hand has no impact to stock's price, and thus they are offered in a separated endpoint -- the Dividend endpoint. In short, if you need all price-impactful corporate split actions, only use the Split endpoint."

The following events change the share count / price scale and therefore require historical price adjustment:
- stock splits
- reverse splits
- stock dividends
- similar share-distribution actions

Those belong in the splits endpoint:
- [Massive Splits Endpoint Docs](https://massive.com/docs/rest/stocks/corporate-actions/splits?utm_source=chatgpt.com)

Ordinary cash dividends do **not** mechanically rescale intraday prices.

Cash dividends are exposed separately here:
- [Massive Dividends Endpoint Docs](https://massive.com/docs/rest/stocks/corporate-actions/dividends?utm_source=chatgpt.com)
#### A cash dividend is a fundamental transfer of value out of the company, not a mathematical restructuring of the shares. Because you are backtesting **intraday** and **swing** strategies using **1-min data**, adjusting historical prices for cash dividends can actually ruin your data. 
Here is how this impacts your specific strategies and how professional quants solve it.
#### Why cash dividends are different
Suppose:
- stock closes at $100
- company pays $1 cash dividend

On ex-dividend date, the market may open near:
- $99

But:
- the exchange did NOT retroactively divide all prior prices by 1%
- the company did NOT change share count
- liquidity/trading mechanics are unchanged

This is economically different from a split.

Suppose:
- stock closes at $100
- overnight:
    - $5 special dividend issued

Next day opens:
- ~$95

If you dividend-adjust minute bars:
- yesterday’s close becomes ~$95 retroactively

Now:
- the overnight gap disappears

But in reality:
- traders experienced a real gap
- overnight risk was real
- options repriced
- stops triggered
- liquidity changed

For many trading systems, removing that is undesirable.
#### The biggest misconception online
A huge amount of retail quant infrastructure inherited conventions from:
- CRSP
- Yahoo Finance
- academic datasets

These are primarily:
- daily-frequency
- total-return-oriented

That convention leaked into:
- GitHub repos
- tutorials
- ML datasets

**But institutional intraday trading systems are usually much more careful.**
#### The Problem with Adjusting for Dividends on Intraday/1-Min Data
When Yahoo Finance or CRSP adjust for cash dividends, they usually apply a backward-looking multiplier. If you apply this multiplier to historical 1-minute data, you create **synthetic prices that never actually existed**.
1. **Tick Size Distortion:** If you proportionally adjust historical 1-minute bars downward, a stock that traded cleanly at $100.00, $100.05, and $100.10 might now show up in your database as $98.31, $98.36, and $98.41. This ruins the simulation of bid/ask spreads and realistic limit order fills.
2. **Loss of Psychological Levels:** Intraday strategies rely heavily on support/resistance, VWAP, and whole-dollar psychological levels (like a breakout at exactly $100.00). If you adjust the historical price down to $98.31, your algorithm will be looking for a breakout at a random, meaningless decimal.
3. **Intraday PnL is Unaffected:** If you buy at 10:00 AM and sell at 3:00 PM, an overnight cash dividend from three months ago is completely irrelevant to that day's price action.
#### For intraday/swing systematic trading, the ex-dividend drop is often **NOT artificial**. It was a real tradable market event. The stock actually traded there.
Your stops, fills, gaps, volatility, overnight risk:
- all experienced that move.

If you retroactively smooth it away:
- you are creating synthetic prices
- that nobody actually traded

That matters enormously for:
- intraday systems
- gap strategies
- stop-based systems
- volatility models
- execution simulation
#### The Problem with NOT Adjusting for Dividends on Swing Data
If you hold a stock overnight (swing trading) through an ex-dividend date:
- Your 1-minute S3 data will show the stock closing at $100 and opening at $99.
- If you don't account for the dividend, your backtesting engine will register a $1 loss.
- In reality, your brokerage account would have received $1 in cash. You didn't lose money; the value just moved from equity to cash. Your backtester needs to know this to accurately calculate your strategy's Sharpe ratio and total return.
#### The institutional reality - Professional systems never rely on one universal “adjusted” series.
Instead they separate:

| Purpose                | Data Type              |
| ---------------------- | ---------------------- |
| Execution research     | Split-adjusted         |
| Intraday signals       | Split-adjusted         |
| Risk modeling          | Usually split-adjusted |
| Portfolio NAV          | Dividend-aware         |
| Total return analytics | Fully adjusted         |
#### The Professional Solution (The "Two-Ledger" Approach)
Since you cannot safely adjust 1-minute historical prices for cash dividends without corrupting the intraday market microstructure, but you must account for them in swing trades, institutional backtesting engines use the following architecture:

**1. The Price Series (What you feed your indicators):**
- Apply **SPLITS ONLY** to your historical 1-minute S3 files.
- This keeps the share count normalized - so a 10-to-1 split doesn't break your chart, but preserves the actual, realistic price levels and tick sizes the stock traded at during that specific period.

**2. The Backtester Accounting (How you calculate PnL):**
- Download the **DIVIDENDS** endpoint, but do not apply it to the price data. Instead, save it as a separate lookup table (a corporate actions calendar).  
- Program your backtesting engine to check this table daily.
- If your swing strategy is holding shares of AAPL at the close of the day before the ex-dividend date, the backtester manually credits your simulated "Cash Balance" with Dividend Amount × Number of Shares Owned.
#### Summary of What You Should Do
- **To build your 1-minute OHLCV database:** Only apply the **Splits endpoint** to adjust prices and volume.    
- **To build your backtesting engine:** Use the **Dividends endpoint** to inject simulated cash payments into your portfolio whenever your swing strategy holds a stock overnight into an ex-dividend date.
#### AND apply adjustments in the order they happened. Otherwise, we get distortions. 
...
#### One more subtle issue: volume adjustment
When adjusting for splits:
You should also adjust:
- volume
- shares traded

Inverse factor.
Example:
- 2-for-1 split
- price ÷ 2
- volume × 2

Many people forget this.
#### Recommended architecture
The cleanest architecture is:
1. Raw data
	- No adjustments.
2. Split-adjusted intraday data
	- Adjust:
		- splits
		- reverse splits
		- stock dividends
	- Do NOT adjust:
		- ordinary cash dividends
	- Use this for:
		- signal generation
		- backtesting
		- execution research
3. Dividend-aware portfolio accounting
	- Separately:
		- add dividend cashflows to PnL/equity curve
	- This preserves:
		- true market prices
		- true execution conditions
		- correct economic returns
4. Optional total-return series
	- Mostly for:
		- long-horizon analytics
		- benchmarking
		- factor research

# 5.1 Downloading adjustments
Adjustment factors are stored in the Polygon folder under <code>raw/adjustments/AAPL.csv</code>. 

### **If a ticker is recycled, then this file has the adjustment factors for all the companies associated with the ticker.** 

To get the adjustments, we simply use the <code>Stock Splits</code> and <code>Dividends</code> endpoint through the SDK. These are <code>list_splits</code> and <code>list_dividends</code>. 

The execution date of a split is when the stock has just been split before market open. So all prices before the split should be adjusted. If the split is 10-to-1, this means that 10 stocks have become 1. So all prices before the execution date must be x10. If the split is 1-to-5, this means 1 stock is split into 5 pieces. Then all prices before the split date have to be divided by 5.

The file with adjustments in the <code>adjustments</code> folder have the following columns: <code>['ticker', 'date', 'type', 'subtype', 'amount']</code> The date is the ex-dividend or execution date. Type is 'DIV' or 'SPLIT'. Subtype is 'CD', 'SD', 'LT', 'ST' for dividends and 'R' (reverse), 'N' (regular) for splits. Amount is the USD amount for dividends and a fraction for splits. A 10-to-1 reverse split is 10. A 1-to-5 split is 0.2.

(NOT NEEDED - Cos we are not accounting for Dividends
Ex-dividend is the date when the investor does *not* get the dividend. If an investor held the stock before, he does. So on ex-dividend date the stock on average drops with the dividend. We need to add the dividends back to the stock price. Or subtract them from before ex-dividend. Most platforms do the backwards adjustments so I will also do that. The advantage is that the current price in the data is then unadjusted and thus equal to the actual market price.
*Note: dividends are [already adjusted](https://polygon.io/knowledge-base/article/does-polygon-adjust-historic-dividends-for-splits) for splits. So we don't have to do this again.*)

# How is this coded?
    # If file exists, add new adjustments. This happens if a ticker is recycled or we are updating and the stock already has a history of adjustments.
    # Dev: No need to overthink it. Even for recycled tickers, adjustments obviously apply to the date in question and hence, the adjustment matches whatever recycled ticker is active at that time.

🗺️ 1. Setup and Configuration
Configuration Flags: The script uses a CLEAN_DOWNLOAD flag, which controls whether to perform a fresh download of all data or to update an existing dataset incrementally.

Date Range: The START_DATE and END_DATE define the window for which data is downloaded. The default range is from September 10, 2003, to April 19, 2024.

API Client: The notebook initializes a Polygon.io REST client using an API key stored in a secret.txt file. This client provides the methods list_splits and list_dividends used to fetch the data.

📥 2. Data Retrieval Process
The script iterates through a list of tickers to fetch data for each one.

Date Filtering: Before making an API call, the script checks if a ticker's listing end_date is before the START_DATE. If so, it skips that ticker to avoid unnecessary API calls for securities that were already delisted.

# ^^^ this is wrong if we are downloading data for backtesting, cos for backtesting we need to apply adjustments to even tickers that are currently delisted so we can backtest on these delisted tickers  too cos at some point historically, they were active

API Calls: For each ticker, it calls:

client.list_splits() to fetch stock split data.

client.list_dividends() to fetch dividend data.

Data Standardization: The raw data from the API is transformed into a consistent schema. The code renames columns, converts date formats, and creates a unified type column ('DIV' for dividends, 'SPLIT' for splits) to make the final dataset uniform.

Calculating Split Factors: For splits, a "split factor" is calculated as split_from / split_to. This factor can then be used to adjust historical prices. For example, a 2-for-1 split would have a factor of 2 / 1 = 2.0, while a 1-for-5 reverse split would be 1 / 5 = 0.2. The script also automatically categorizes the split subtype as either 'R' for reverse or 'N' for a normal split.

🛠️ 3. Data Processing and Deduplication
After fetching data, the script performs several processing steps.

Combining Data: The dividend and split DataFrames for a given ticker are concatenated and then sorted by date.

Deduplication: The script includes a crucial deduplication step to prevent errors. It checks for and removes duplicate entries, first for dividends, then for splits. This safeguards against potential issues caused by ticker recycling or API quirks.

💾 4. Saving Data
The processed data is saved to a CSV file for each ticker, using the ticker symbol as the filename (e.g., AAPL.csv). It is stored in a structured directory: POLYGON_DATA_PATH/raw/adjustments/.

Update Logic: The script elegantly handles updates. If a file for a ticker already exists, it reads the old data, concatenates it with the new data, and then saves the combined, deduplicated result back to the same file.

Tracking Progress: To facilitate incremental updates, the script writes the END_DATE of the current run to a _end_date.txt file. On subsequent runs with CLEAN_DOWNLOAD=False, the script reads this file to determine a new START_DATE, resuming the download from where it left off.

💡 Key Considerations and Context
Why "Backwards Adjustments"?
The script's goal is to enable "backwards" adjustments to historical prices. This means that if you multiply all past prices by a cumulative adjustment factor, the current price remains unchanged, equaling the actual market price. This approach is widely used because it ensures the most recent data is always accurate and does not need to be recalculated every time an adjustment happens.

A Crucial Detail on Dividends
The notebook's comments include an important note that dividends returned by Polygon's API are already adjusted for splits. This means the script must be careful not to apply a split adjustment to a dividend event that has already been accounted for.

Handling Ticker Recycling
A single ticker symbol can be reused by different companies over time. The code accounts for this by appending new adjustment data to the existing CSV file for that ticker, which is why it also contains deduplication logic to manage the data correctly.

In [1]:
from massive.rest import RESTClient
from datetime import datetime, date, time, timedelta
from times import first_trading_date_after_equal, last_trading_date_before_equal
from tickers import get_tickers
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

client = RESTClient(api_key=API_KEY)

POLYGON_DATA_PATH = "../data/polygon/"

START_DATE = date(2016, 6, 1) # only go as far as your polygon membership allows. don't waste loop cycles
END_DATE = date(2026, 6, 15) # till today only or else you will get "out of range" from our custom function and don't waste loop cycles
# END_DATE = this will need updating everyday so that we only fetch the new adjustments in an idempotent way

# CLEAN_DOWNLOAD flag controls whether to perform a fresh download of all data or to update an existing dataset incrementally.
CLEAN_DOWNLOAD = True # If False, only update existing data to the END_DATE. If True, remove all files in adjustments/.

# Polygon dividends are NOT adjusted for splits! Verified below.
## It's crucial to adjust Dividends for Splits!
https://massive.com/knowledge-base/article/does-massive-adjust-historic-dividends-for-splits


Why this is crucial:

https://aistudio.google.com/app/prompts?state=%7B%22ids%22:%5B%221CmuSHfJT0xy4UGGdELP5RLbrclRjvgiR%22%5D,%22action%22:%22open%22,%22userId%22:%22104962803681652166866%22,%22resourceKeys%22:%7B%7D%7D&usp=sharing

### Verifying whether Dividends are adjusted for Split already or not.
The column likely_adjusted will be True if the actual ratio is closer to 1.0, False if it is closer to the split factor.

If you see several splits where the ratio is near the split factor, Polygon.io is not adjusting historic dividends for splits.

If the ratio is consistently near 1.0, then the notebook’s original assumption is correct and the data is already split‑adjusted.

Run this on a handful of liquid stocks with known splits; the pattern should be clear almost immediately.

**This fetches both dividends and splits for a ticker, then checks whether the pre‑split dividend amounts have been scaled by the split factor.**

In [30]:
# Helper functions to fetch data from Massive
def get_dividends(ticker: str, limit: int = 500) -> pd.DataFrame:
    """
    Fetch dividend history from Massive.
    Massive’s /v2/dividends returns a list of dicts.
    We normalise into a DataFrame with datetime columns.
    """
    try:
        data = client.list_dividends(ticker=ticker, limit=limit)  # returns list of dicts
    except AttributeError:
        # Fallback to raw request if the high‑level method is not available
        resp = client.get(f"/v2/dividends/{ticker}", params={"limit": limit})
        data = resp.json()
        if isinstance(data, dict):
            data = data.get("results", [])

    df = pd.DataFrame(data)
    if df.empty:
        return df

    # Normalise column names (Massive uses snake_case; dates as strings)
    df.rename(columns={
        "ex_dividend_date": "ex_dividend_date",
        "pay_date": "pay_date",
        "cash_amount": "cash_amount",
    }, inplace=True)

    df["ex_dividend_date"] = pd.to_datetime(df["ex_dividend_date"])
    if "pay_date" in df.columns:
        df["pay_date"] = pd.to_datetime(df["pay_date"])
    df = df.sort_values("ex_dividend_date")
    # print(df)
    return df


def get_splits(ticker: str, limit: int = 100) -> pd.DataFrame:
    """
    Fetch stock split history from Massive.
    """
    try:
        data = client.list_splits(ticker=ticker, limit=limit)
    except AttributeError:
        resp = client.get(f"/v2/splits/{ticker}", params={"limit": limit})
        data = resp.json()
        if isinstance(data, dict):
            data = data.get("results", [])

    df = pd.DataFrame(data)
    if df.empty:
        return df

    df.rename(columns={
        "execution_date": "execution_date",
        "split_from": "split_from",
        "split_to": "split_to",
    }, inplace=True)

    df["execution_date"] = pd.to_datetime(df["execution_date"])
    df = df.sort_values("execution_date")
    return df

# ### 3. Analysis logic
def check_adjustment(ticker: str) -> pd.DataFrame:
    """
    For a given ticker, examine each split and compare the last pre‑split dividend
    with the first post‑split dividend.
    """
    divs = get_dividends(ticker)
    splits = get_splits(ticker)

    if splits.empty:
        print(f"No splits found for {ticker}")
        return pd.DataFrame()

    results = []
    for _, split in splits.iterrows():
        split_date = split["execution_date"]
        from_shares = int(split["split_from"])
        to_shares = int(split["split_to"])
        factor = to_shares / from_shares   # >1 for a forward split (e.g. 2:1 → 2)

        pre_divs = divs[divs["ex_dividend_date"] < split_date].tail(1)
        # print(pre_divs)
        post_divs = divs[divs["ex_dividend_date"] >= split_date].head(1)
        # print(post_divs)

        if pre_divs.empty or post_divs.empty:
            continue

        pre_amt = pre_divs.iloc[0]["cash_amount"]
        post_amt = post_divs.iloc[0]["cash_amount"]
        if post_amt == 0:
            continue

        ratio = pre_amt / post_amt

        results.append({
            "split_date": split_date.date(),
            "from -> to": f"{from_shares} -> {to_shares}",
            "factor": factor,
            "pre_div_date": pre_divs.iloc[0]["ex_dividend_date"].date(),
            "pre_amount": pre_amt,
            "post_div_date": post_divs.iloc[0]["ex_dividend_date"].date(),
            "post_amount": post_amt,
            "ratio": ratio,
            "expected_if_unadjusted": factor,
            "expected_if_adjusted": 1.0,
            "likely_adjusted": abs(ratio - 1.0) < abs(ratio - factor),
        })

    return pd.DataFrame(results)

In [ ]:
# ### 4. Run on a few classic split/dividend stocks

# Why ratio ≈ split factor means the data is NOT adjusted
# The pre_amount is the dividend as it was originally paid (e.g., $0.82 per pre‑split share).
# The post_amount is the dividend after the split, reflecting the new per‑share reality (e.g., $0.205).
# If the data were split‑adjusted, the pre‑split dividend would have been restated to $0.205 in the database (so that all dividends are expressed on the same per‑share basis). 
# Then pre/post would be 1.0, not 4.0.
# Since you see pre/post == 4.0, the pre‑split amount has not been scaled down – it’s still the original, unadjusted value.
# Every one of your tickers shows that Massive/Polygon dividends are NOT adjusted for splits. The likely_adjusted column is correctly False.

tickers_to_test = ["WMT", "AAPL", "AVGO", "FAST", "TSCO"]

for tkr in tickers_to_test:
    print(f"\n--- {tkr} ---")
    res = check_adjustment(tkr)
    if not res.empty:
        display(res[[
            "split_date", "from -> to", "pre_amount", "post_amount",
            "ratio", "expected_if_unadjusted", "expected_if_adjusted", "likely_adjusted"
        ]])


--- WMT ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-02-26,1 -> 3,0.57,0.2075,2.746988,3.0,1.0,False



--- AAPL ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2014-06-09,1 -> 7,3.29,0.470,7.0,7.0,1.0,False
1,2020-08-31,1 -> 4,0.82,0.205,4.0,4.0,1.0,False



--- AVGO ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-07-15,1 -> 10,5.25,0.53,9.90566,10.0,1.0,False



--- FAST ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2005-11-14,1 -> 2,0.31,0.20,1.550000,2.0,1.0,False
1,2011-05-23,1 -> 2,0.26,0.13,2.000000,2.0,1.0,False
2,2019-05-23,1 -> 2,0.43,0.22,1.954545,2.0,1.0,False
3,2025-05-22,1 -> 2,0.44,0.22,2.000000,2.0,1.0,False



--- TSCO ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2010-09-03,1 -> 2,0.14,0.07,2.000000,2.0,1.0,False
1,2013-09-27,1 -> 2,0.26,0.13,2.000000,2.0,1.0,False
2,2024-12-20,1 -> 5,1.10,0.23,4.782609,5.0,1.0,False


# Getting Split and Dividend Adjustments for ALL tickers

In [2]:
# TO BE DELETED
""" # Loop through all tickers
tickers_v1 = get_tickers(v=3)

# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.

# Use a different START_DATE if we only want to update
if not CLEAN_DOWNLOAD:
    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):
        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:
            end_date_of_data = next(f).strip()
            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))
    else:
        raise Exception('There is no _end_date.txt file!')
        
for index, row in tickers_v1.iterrows():
    if index <= 5739:
        continue
    ticker = row["ticker"]
    end_date = row["end_date"]

    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.
    if end_date < START_DATE:
        continue

    # Get data
    try:
        splits = pd.DataFrame(client.list_splits(ticker=ticker, execution_date_gte=START_DATE, execution_date_lte=END_DATE))
        dividends = pd.DataFrame(client.list_dividends(ticker=ticker, ex_dividend_date_gte=START_DATE, ex_dividend_date_lte =END_DATE))
    except Exception as e:
        print(repr(e))
        continue

    # Get correct format
    if not dividends.empty:
        dividends = dividends.rename(columns={'ex_dividend_date': 'date', 'dividend_type':'subtype', 'cash_amount': 'amount'})
        dividends['type'] = 'DIV'
        dividends = dividends[['date', 'type', 'subtype', 'amount']]

    if not splits.empty:
        splits = splits.rename(columns={'execution_date': 'date'})
        splits['type'] = 'SPLIT'
        splits['subtype'] = np.where(splits['split_from'] > splits['split_to'], 'R', 'N')
        splits['amount'] = splits['split_from'] / splits['split_to']
        splits = splits[['date', 'type', 'subtype', 'amount']]
    
    # Skip loop if no data
    if splits.empty and dividends.empty:
        continue

    # Merge dividends and splits
    adjustments = pd.concat([dividends, splits])
    adjustments = adjustments.sort_values(by='date').reset_index(drop=True)
    adjustments['date'] = pd.to_datetime(adjustments['date']).dt.date

    if adjustments.isnull().values.any():
        null_data = tickers_v1[tickers_v1[["ticker", "name", "active", "type", "start_date", "last_updated_utc"]].isnull().any(axis=1)]
        raise Exception(f"There are missing values for {ticker} at index {index}.")

    # Save or update
    path = POLYGON_DATA_PATH + f'raw/adjustments/{ticker}.csv'

    # If file exists, add new adjustments. This happens if a ticker is recycled or we are updating and the stock already has a history of adjustments.
    # Dev: No need to overthink it. Even for recycled tickers, adjustments obviously apply to the date in question and hence, 
    # the adjustment matches whatever recycled ticker is active at that time.
    if os.path.isfile(path):
        old_adjustments = pd.read_csv(
            path,
            parse_dates=True,
        )
        all_adjustments = pd.concat([old_adjustments, adjustments])
        all_adjustments['date'] = pd.to_datetime(all_adjustments['date']).dt.date
        all_adjustments = all_adjustments.sort_values(by='date').reset_index(drop=True)
        
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not all_adjustments[all_adjustments['type'] == 'DIV'].empty:
            dividends = all_adjustments[all_adjustments['type'] == 'DIV']
            indices_to_remove = dividends[dividends[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        if not all_adjustments[all_adjustments['type'] == 'SPLIT'].empty:
            splits = all_adjustments[all_adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
            
        all_adjustments.to_csv(path, index=False)
    else:
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not all_adjustments[all_adjustments['type'] == 'DIV'].empty:
            dividends = all_adjustments[all_adjustments['type'] == 'DIV']
            indices_to_remove = dividends[dividends[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        if not all_adjustments[all_adjustments['type'] == 'SPLIT'].empty:
            splits = all_adjustments[all_adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        adjustments.to_csv(path, index=False)

# Set the END_DATE to end_date.txt
with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt", 'w') as f:
    f.write(f'{END_DATE}\n') """

' # Loop through all tickers\ntickers_v1 = get_tickers(v=3)\n\n# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.\n\n# Use a different START_DATE if we only want to update\nif not CLEAN_DOWNLOAD:\n    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):\n        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:\n            end_date_of_data = next(f).strip()\n            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))\n    else:\n        raise Exception(\'There is no _end_date.txt file!\')\n\nfor index, row in tickers_v1.iterrows():\n    if index <= 5739:\n        continue\n    ticker = row["ticker"]\n    end_date = row["end_date"]\n\n    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.\n    if end_date < START_DATE:\n        continue\n\n    # Get data\n    try:\n        splits = pd.DataFra

In [3]:
""" # Loop through all tickers
tickers_v1 = get_tickers(v=1)
# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.

# Use a different START_DATE if we only want to update
if not CLEAN_DOWNLOAD:
    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):
        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:
            end_date_of_data = next(f).strip()
            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))
    else:
        raise Exception('There is no _end_date.txt file!')
        
for index, row in tickers_v1.iterrows():
    # if index <= 5739:
    if index <= 50000:
        continue
    ticker = row["ticker"]
    end_date = row["end_date"]

    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.
    if end_date < START_DATE:
        continue

    # Get data
    try:
        splits = pd.DataFrame(client.list_splits(ticker=ticker, execution_date_gte=START_DATE, execution_date_lte=END_DATE))
    except Exception as e:
        print(repr(e))
        continue

    # Get correct format
    if not splits.empty:
        splits = splits.rename(columns={'execution_date': 'date'})
        splits['type'] = 'SPLIT'
        splits['subtype'] = np.where(splits['split_from'] > splits['split_to'], 'R', 'N')
        splits['amount'] = splits['split_from'] / splits['split_to']
        splits = splits[['date', 'type', 'subtype', 'amount']]
    
    # Skip loop if no data
    if splits.empty:
        continue

    # Merge dividends and splits (NOT NEEDED)
    # adjustments = pd.concat([dividends, splits])
    adjustments = splits
    adjustments = adjustments.sort_values(by='date').reset_index(drop=True)
    adjustments['date'] = pd.to_datetime(adjustments['date']).dt.date

    if adjustments.isnull().values.any():
        null_data = tickers_v1[tickers_v1[["ticker", "name", "active", "type", "start_date", "last_updated_utc"]].isnull().any(axis=1)]
        raise Exception(f"There are missing values for {ticker} at index {index}.")

    # Save or update
    path = POLYGON_DATA_PATH + f'raw/adjustments/{ticker}.csv'

    # If file exists, add new adjustments. This happens if a ticker is recycled or we are updating and the stock already has a history of adjustments.
    # Dev: No need to overthink it. Even for recycled tickers, adjustments obviously apply to the date in question and hence, 
    # the adjustment matches whatever recycled ticker is active at that time.
    if os.path.isfile(path):
        old_adjustments = pd.read_csv(
            path,
            parse_dates=True,
        )
        all_adjustments = pd.concat([old_adjustments, adjustments])
        all_adjustments['date'] = pd.to_datetime(all_adjustments['date']).dt.date
        all_adjustments = all_adjustments.sort_values(by='date').reset_index(drop=True)
        
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not all_adjustments[all_adjustments['type'] == 'SPLIT'].empty:
            splits = all_adjustments[all_adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        all_adjustments.to_csv(path, index=False)
    else:
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not adjustments[adjustments['type'] == 'SPLIT'].empty:
            splits = adjustments[adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            adjustments = adjustments.drop(index=indices_to_remove)
        adjustments.to_csv(path, index=False)

# Set the END_DATE to end_date.txt
with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt", 'w') as f:
    f.write(f'{END_DATE}\n') """

' # Loop through all tickers\ntickers_v1 = get_tickers(v=1)\n# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.\n\n# Use a different START_DATE if we only want to update\nif not CLEAN_DOWNLOAD:\n    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):\n        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:\n            end_date_of_data = next(f).strip()\n            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))\n    else:\n        raise Exception(\'There is no _end_date.txt file!\')\n\nfor index, row in tickers_v1.iterrows():\n    # if index <= 5739:\n    if index <= 50000:\n        continue\n    ticker = row["ticker"]\n    end_date = row["end_date"]\n\n    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.\n    if end_date < START_DATE:\n        continue\n\n    # Get data\n    try:\n   

In [4]:
# Loop through all tickers
tickers_v1 = get_tickers(v=1)
# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.

# Updates only
# Use a different START_DATE if we only want to update
if not CLEAN_DOWNLOAD:
    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):
        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:
            end_date_of_data = next(f).strip()
            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))
    else:
        raise Exception('There is no _end_date.txt file!')
        
for index, row in tickers_v1.iterrows():
    # if index <= 5739:
    #     continue
    ticker = row["ticker"]
    end_date = row["end_date"]

    # WRONG! Below is wrong if we are downloading data for backtesting, cos for backtesting we need to apply adjustments to 
    # even tickers that are currently delisted so we can backtest on these delisted tickers  too cos 
    # at some point historically they were active.
    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.
    # if end_date < START_DATE:
    #     continue

    # Get data
    try:
        splits = pd.DataFrame(client.list_splits(ticker=ticker, execution_date_gte=START_DATE, execution_date_lte=END_DATE))
    except Exception as e:
        print(repr(e))
        continue

    # Get correct format
    if not splits.empty:
        splits = splits.rename(columns={'execution_date': 'date'})
        splits['type'] = 'SPLIT'
        splits['subtype'] = np.where(splits['split_from'] > splits['split_to'], 'R', 'N')
        splits['amount'] = splits['split_from'] / splits['split_to']
        splits = splits[['date', 'type', 'subtype', 'amount']]
    
    # Skip loop if no data
    if splits.empty:
        continue

    # Merge dividends and splits (NOT NEEDED)
    # adjustments = pd.concat([dividends, splits])
    adjustments = splits
    adjustments = adjustments.sort_values(by='date').reset_index(drop=True)
    adjustments['date'] = pd.to_datetime(adjustments['date']).dt.date

    if adjustments.isnull().values.any():
        null_data = tickers_v1[tickers_v1[["ticker", "name", "active", "type", "start_date", "last_updated_utc"]].isnull().any(axis=1)]
        raise Exception(f"There are missing values for {ticker} at index {index}.")

    # Save or update
    path = POLYGON_DATA_PATH + f'raw/adjustments/{ticker}.csv'

    # If file exists, add new adjustments. This happens if a ticker is recycled or we are updating and the stock already has a history of adjustments.
    # Dev: No need to overthink it. Even for recycled tickers, adjustments obviously apply to the date in question and hence, 
    # the adjustment matches whatever recycled ticker is active at that time.
    if os.path.isfile(path):
        old_adjustments = pd.read_csv(
            path,
            parse_dates=True,
        )
        all_adjustments = pd.concat([old_adjustments, adjustments])
        all_adjustments['date'] = pd.to_datetime(all_adjustments['date']).dt.date
        all_adjustments = all_adjustments.sort_values(by='date').reset_index(drop=True)
        
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not all_adjustments[all_adjustments['type'] == 'SPLIT'].empty:
            splits = all_adjustments[all_adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        all_adjustments.to_csv(path, index=False)
    else:
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not adjustments[adjustments['type'] == 'SPLIT'].empty:
            splits = adjustments[adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            adjustments = adjustments.drop(index=indices_to_remove)
        adjustments.to_csv(path, index=False)

# Set the END_DATE to end_date.txt
with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt", 'w') as f:
    f.write(f'{END_DATE}\n')

In [8]:
tickers_v1[tickers_v1['ticker'] == 'ESCA']

,ID,ticker,name,active,start_date,end_date,type,cik,composite_figi
3504,ESCA-2016-06-06,ESCA,Escalade Inc,True,2016-06-06,2026-06-03,CS,33488.0,BBG000BJ0H70


Some examples.

In [6]:
pd.read_csv(POLYGON_DATA_PATH + "raw/adjustments/MFA.csv", index_col="date").tail(7)

,type,subtype,amount
date,,,
2022-04-05,SPLIT,R,4.0


In [7]:
pd.read_csv(POLYGON_DATA_PATH + "raw/adjustments/TSLA.csv", index_col="date")

,type,subtype,amount
date,,,
2020-08-31,SPLIT,N,0.200000
2022-08-25,SPLIT,N,0.333333


### Updates
Simply rerun the file with the correct <code>END_DATE</code> and set <code>CLEAN_DOWNLOAD</code> to False. The code is already structured in such a way that only the necessary is downloaded. Also, dividends have to be split adjusted. The code is below.